Link: https://www.kaggle.com/datasets/shub99/sentiment-analysis-data

In [16]:
import pandas as pd
from sklearn.datasets import load_files
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report
import nltk
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec

## EDA

Let's read the date set with reviews 

In [17]:
df = pd.read_csv('Restaurant_Reviews.tsv', sep='\t')

Data size

The first 5 rows of data.

1 - a good feedback

0- bad feedback

In [18]:
print(df.shape)
print(df.head(5))

(1000, 2)
                                              Review  Liked
0                           Wow... Loved this place.      1
1                                 Crust is not good.      0
2          Not tasty and the texture was just nasty.      0
3  Stopped by during the late May bank holiday of...      1
4  The selection on the menu was great and so wer...      1


Let’s analyse how many negative and positive reviews there are on DataSet

In [19]:
df['Liked'].value_counts()

Liked
1    500
0    500
Name: count, dtype: int64

As we can see, there are an equal number of negative and positive reviews (500 positive and 500 negative reviews), which means that the ‘Accuracy’ metric can be used as an appropriate indicator of model quality 

Let’s look at some examples of negative and positive reviews.

Позитивні

In [20]:
for r in df[df['Liked'] == 1]['Review'].head(5):
    print(r)

Wow... Loved this place.
Stopped by during the late May bank holiday off Rick Steve recommendation and loved it.
The selection on the menu was great and so were the prices.
The fries were great too.
A great touch.


Негативні

In [21]:
for r in df[df['Liked'] == 0]['Review'].head(5):
    print(r)

Crust is not good.
Not tasty and the texture was just nasty.
Now I am getting angry and I want my damn pho.
Honeslty it didn't taste THAT fresh.)
The potatoes were like rubber and you could tell they had been made up ahead of time being kept under a warmer.


We divide the data set into a training set and a test set

In [22]:
train_df, test_df = train_test_split(df, test_size=0.25, random_state=42)

In [23]:
x_train, y_train = train_df['Review'], train_df['Liked'] 
x_test, y_test = test_df['Review'], test_df['Liked'] 


In [24]:
print(f"Quantity for training purposes: {len(x_train)}")
print(f"Quantity for testing: {len(x_test)}")

Quantity for training purposes: 750
Quantity for testing: 250


In [25]:
print("Samples per class (training): {}".format(np.bincount(y_train)))

Samples per class (training): [372 378]


### Bag-of-word

In [26]:
vect = CountVectorizer().fit(x_train)
Xx_train = vect.transform(x_train)
print('x_train: \n{}'.format(repr(Xx_train)))

x_train: 
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 7387 stored elements and shape (750, 1728)>


In [27]:
print("Size of the obtained matrix:", x_train.shape)

Size of the obtained matrix: (750,)


In [28]:
print("Samples per class: {}".format(np.bincount(y_train)))

Samples per class: [372 378]


In [29]:
print("Vocabulary size: {}".format(len(vect.vocabulary_)))
print("Vocabulary content:\n {}".format(list(vect.vocabulary_.keys())[5:10]))

Vocabulary size: 1728
Vocabulary content:
 ['were', 'bland', 'spend', 'your', 'money']


In [30]:
feature_names = vect.get_feature_names_out()
print("Number of features: {}".format(len(feature_names)))
print("First 20 features:\n{}".format(feature_names[:20]))
print("Features 210 to 230:\n{}".format(feature_names[210:230]))
print("Every 750th feature:\n{}".format(feature_names[::750]))

Number of features: 1728
First 20 features:
['10' '100' '12' '17' '1979' '20' '2007' '30' '30s' '35' '40' '40min' '45'
 '4ths' '5lb' '70' '85' '90' 'about' 'above']
Features 210 to 230:
['buldogis' 'bunch' 'burger' 'burgers' 'burrittos' 'bus' 'business'
 'bussell' 'busy' 'but' 'butter' 'by' 'bye' 'caballero' 'caesar' 'cafe'
 'café' 'calamari' 'call' 'calligraphy']
Every 750th feature:
['10' 'horrible' 'tepid']


In [31]:
scores = cross_val_score(LogisticRegression(), Xx_train, y_train, cv=4)
print("Mean cross-validation accuracy: {:.2f}".format(np.mean(scores)))

Mean cross-validation accuracy: 0.82


### Training logistic regression with hyperparameter tuning

We use **Logistic Regression** to classify sentiment (positive/negative review). To achieve the best accuracy, we use GridSearchCV, which automatically finds the optimal regularisation parameter C.

* **Cross-validation (cv=5)**: splitting the data into 5 parts to objectively assess the model’s stability.
* **Parameter C**: controlling the model’s complexity to prevent overfitting.

In [32]:
param_grid = {'C': [0.001, 0.01, 0.1, 4, 3, 5]}
grid = GridSearchCV(LogisticRegression(), param_grid, cv=4)
grid.fit(Xx_train, y_train)
print("Best cross-validation score: {:.2f}".format(grid.best_score_))
print("Best parameters: ", grid.best_params_)

Best cross-validation score: 0.82
Best parameters:  {'C': 5}


In [33]:
xx_test = vect.transform(x_test)
print("{:.2f}".format(grid.score(xx_test, y_test)))

0.82


Stopwords

In [34]:
print('Number of stop words: {}'.format(len(ENGLISH_STOP_WORDS)))
print("Every 10th stopword:\n{}".format(list(ENGLISH_STOP_WORDS)[::10]))

Number of stop words: 318
Every 10th stopword:
['nobody', 'call', 'as', 'whom', 'then', 'becomes', 'cannot', 'thereupon', 'back', 'fill', 'could', 'enough', 'throughout', 'there', 'sincere', 'own', 'most', 'anyway', 'would', 'front', 'his', 'where', 'latterly', 'been', 'still', 'every', 'somehow', 'this', 'well', 'above', 'under', 'whole']


In [51]:
Vect = CountVectorizer(min_df=5, stop_words='english').fit(x_train)
XX_train = Vect.transform(x_train)
print('X_train with stop words:\n{}'.format(repr(x_train[:10])))

X_train with stop words:
82                 The turkey and roast beef were bland.
991           Spend your money and time some place else.
789         Great atmosphere, friendly and fast service.
894    If you stay in Vegas you must get breakfast he...
398    Service is perfect and the family atmosphere i...
323    A couple of months later, I returned and had a...
519    Point your finger at any item on the menu, ord...
916    There is really nothing for me at postinos, ho...
922    Every time I eat here, I see caring teamwork t...
5         Now I am getting angry and I want my damn pho.
Name: Review, dtype: str


In [36]:
y_pred_bow = grid.predict(xx_test)
print("BoW")
print(classification_report(y_test, y_pred_bow))

BoW
              precision    recall  f1-score   support

           0       0.81      0.84      0.82       128
           1       0.83      0.79      0.81       122

    accuracy                           0.82       250
   macro avg       0.82      0.82      0.82       250
weighted avg       0.82      0.82      0.82       250



In [37]:
grid = GridSearchCV(LogisticRegression(), param_grid, cv=4)
grid.fit(XX_train, y_train)
print("Best cross-validation score: {:.2f}".format(grid.best_score_))

Best cross-validation score: 0.77


### Analysis of the BoW model results

The accuracy achieved in cross-validation is **0.82** (or 82 per cent). This indicates that even a simple word-counting method can effectively distinguish between positive and negative reviews.

**Key observations:**
* The use of stop words and a minimum frequency threshold (min_df=5) reduced the size of the vocabulary, which, on this dataset, led to a slight decrease in accuracy compared to the baseline BoW model.
* The model performs consistently on a small dataset (1,000 reviews).

This result will subsequently be used as a **baseline** for comparison with more complex methods, such as TF-IDF and Word2Vec.

### TF-IDF (term frequency-inverse document frequency)

In [38]:
pipe = make_pipeline(TfidfVectorizer(min_df=5, norm=None), LogisticRegression())
param_grid = {'logisticregression__C': [0.001, 0.01, 0.1, 1, 10]}
grid = GridSearchCV(pipe, param_grid, cv=5)
grid.fit(x_train, y_train)
print(grid.best_params_)
print("Best cross-validation score: {:.2f}".format(grid.best_score_))

{'logisticregression__C': 0.01}
Best cross-validation score: 0.76


In [39]:
vectorizer = grid.best_estimator_.named_steps["tfidfvectorizer"]
X_Train = vectorizer.transform(x_train)
max_value = X_Train.max(axis=0).toarray().ravel()
sorted_by_tfidf = max_value.argsort()
feature_names = np.array(vectorizer.get_feature_names_out())
print("Features with lowest tfidf:\n{}".format(
feature_names[sorted_by_tfidf[:20]]))
print("Features with highest tfidf: \n{}".format(
feature_names[sorted_by_tfidf[-20:]]))

Features with lowest tfidf:
['service' 'my' 'but' 'have' 'at' 'here' 'time' 'go' 'an' 'one' 'don'
 'just' 'out' 'your' 'delicious' 'nice' 'only' 'would' 'came' 'ever']
Features with highest tfidf: 
['night' 'she' 'burger' 'sandwich' 'sauce' 'probably' 'flavor' 'slow'
 'dishes' 'meal' 'selection' 'dining' 'you' 'area' 'another' 'once'
 'steak' 'wait' 'waited' 'said']


Analysis of IDF (Inverse Document Frequency) metrics

The **IDF** metric allows us to assess the informativeness of words. The lower the IDF value, the more frequently the word appears across all reviews in the corpus. This helps to identify ‘stop words’ specific to a particular dataset (for example, the words ‘food’ or ‘place’, which appear in most restaurant reviews).

Sorting features in ascending order of IDF allows you to identify the most common terms, which carry the least weight in classification.

In [40]:
sorted_by_idf = np.argsort(vectorizer.idf_)
print("Features with lowest idf:\n{}".format(
feature_names[sorted_by_idf[:100]]))

Features with lowest idf:
['the' 'and' 'was' 'to' 'is' 'it' 'this' 'of' 'food' 'not' 'in' 'place'
 'for' 'good' 'we' 'service' 'had' 'my' 'great' 'so' 'that' 'with' 'be'
 'back' 'very' 'but' 'were' 'have' 'at' 'you' 'are' 'here' 'on' 'they'
 'like' 'all' 'time' 'really' 'our' 'will' 'go' 'there' 'as' 'best' 'an'
 'one' 'don' 'their' 'out' 'just' 'delicious' 'your' 'nice' 'up' 'would'
 'only' 'friendly' 'ever' 'did' 'came' 'also' 'by' 'can' 'restaurant' 'no'
 'again' 'when' 'amazing' 'vegas' 'if' 'us' 'love' 'what' 'won'
 'experience' 'first' 'pretty' 'never' 'salad' 'say' 'from' 'going' 'even'
 'too' 'been' 'disappointed' 'definitely' 'eat' 'me' 'much' 'staff' 'he'
 'get' 'got' 'bad' 'about' 'because' 'which' 'always' 'more']


In [41]:
cv = CountVectorizer(ngram_range=(1, 1)).fit(x_train)
print("Vocabulary size: {}".format(len(cv.vocabulary_)))
print("Vocabulary:\n{}".format(cv.get_feature_names_out()[50:60]))

Vocabulary size: 1728
Vocabulary:
['amount' 'ample' 'an' 'and' 'andddd' 'angry' 'another' 'any' 'anymore'
 'anyone']


In [42]:
pipee = make_pipeline(TfidfVectorizer(min_df=5), LogisticRegression())
paramm_grid = {"logisticregression__C": [0.001, 0.01, 0.1, 1, 10, 100],
"tfidfvectorizer__ngram_range": [(1, 1), (1, 2), (1, 3)]}

gridd = GridSearchCV(pipee, paramm_grid, cv=5)
gridd.fit(x_train, y_train)
print("Best cross-validation score: {:.2f}".format(gridd.best_score_))
print("Best parameters:\n{}".format(gridd.best_params_))

Best cross-validation score: 0.77
Best parameters:
{'logisticregression__C': 1, 'tfidfvectorizer__ngram_range': (1, 1)}


In [43]:

y_pred_tfidf = gridd.best_estimator_.predict(x_test)
print("TF-IDF")
print(classification_report(y_test, y_pred_tfidf))

TF-IDF
              precision    recall  f1-score   support

           0       0.78      0.81      0.79       128
           1       0.79      0.75      0.77       122

    accuracy                           0.78       250
   macro avg       0.78      0.78      0.78       250
weighted avg       0.78      0.78      0.78       250



### Word2Vec

In [44]:
def tokenizer(text):
    tokens = word_tokenize(text.lower())
    return [t for t in tokens if t.isalpha()]

x_train_tokens = [tokenizer(text) for text in x_train]
x_test_tokens = [tokenizer(text) for text in x_test]

In [45]:
model = Word2Vec(sentences=x_train_tokens, vector_size=100, window=5, min_count=1, workers=4)


In [46]:
word_vectors = model.wv

In [47]:

def get_avg_vector(tokens, model, size):
    vec = np.zeros(size)
    count = 0
    for word in tokens:
        if word in model.wv:
            vec += model.wv[word]
            count += 1
    return vec / count if count > 0 else vec

X_train_w2v = np.array([get_avg_vector(t, model, 100) for t in x_train_tokens])
X_test_w2v  = np.array([get_avg_vector(t, model, 100) for t in x_test_tokens])


In [48]:
grid_w2v = GridSearchCV(LogisticRegression(max_iter=1000), {'C': [0.1, 1, 10, 100]}, cv=5)
grid_w2v.fit(X_train_w2v, y_train)
print("Accuracy Word2Vec:", grid_w2v.best_score_)

Accuracy Word2Vec: 0.6066666666666667


In [49]:

y_pred = grid_w2v.predict(X_test_w2v)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.67      0.48      0.56       128
           1       0.58      0.75      0.65       122

    accuracy                           0.61       250
   macro avg       0.62      0.62      0.61       250
weighted avg       0.62      0.61      0.61       250



In [50]:
print("A comparison of methods")
print(f"BoW (baseline):    {np.mean(scores):.2f}")
print(f"TF-IDF + n-grams:  {gridd.best_score_:.2f}")
print(f"Word2Vec:          {grid_w2v.best_score_:.2f}")

A comparison of methods
BoW (baseline):    0.82
TF-IDF + n-grams:  0.77
Word2Vec:          0.61
